# 相机 3D 位置计算器

从单打场 4 个角分别量到相机的激光斜距，用多点定位算出相机精确 3D 坐标。

```
       角1 (1.37, 0)────────角2 (6.86, 0)       近端底线
           │                    │
           │       球场         │
           │                    │
       角3 (1.37, 23.77)───角4 (6.86, 23.77)    远端底线
```

In [1]:
import numpy as np
from scipy.optimize import least_squares

# ============================================================
# ★ 在这里填入你的激光测距数据（单位：米）
# ============================================================

# 单打场 4 个角的世界坐标 (x, y, z=0)
corners = np.array([
    [1.37,  0.00, 0],   # 角1: 近端左
    [6.86,  0.00, 0],   # 角2: 近端右
    [1.37, 23.77, 0],   # 角3: 远端左
    [6.86, 23.77, 0],   # 角4: 远端右
])

# --- cam66 (近端相机) ---
# 从每个角量到 cam66 镜头的斜距（米）
cam66_distances = [
    30.0,   # d1: 角1(近端左) → cam66
    29.938,   # d2: 角2(近端右) → cam66
    9.164,   # d3: 角3(远端左) → cam66
    8.964,   # d4: 角4(远端右) → cam66

]

# --- cam68 (远端相机) ---
# 从每个角量到 cam68 镜头的斜距（米）
cam68_distances = [
    8.591,   # d5: 角1(近端左) → cam68
    8.454,   # d6: 角2(近端右) → cam68
    29.954,   # d7: 角3(远端左) → cam68
    29.664,   # d8: 角4(远端右) → cam68


]

print('请填入测量数据后重新运行此 cell')
print(f'cam66 斜距: {cam66_distances}')
print(f'cam68 斜距: {cam68_distances}')

请填入测量数据后重新运行此 cell
cam66 斜距: [30.0, 29.938, 9.164, 8.964]
cam68 斜距: [8.591, 8.454, 29.954, 29.664]


In [3]:
# ============================================================
# 求解相机 3D 位置
# ============================================================

def solve_camera_position(corners, distances, name, initial_guess=None):
    """从 4 个已知点的斜距求解相机 3D 位置。
    
    方程: di² = (cx - xi)² + (cy - yi)² + (cz - 0)²
    4 个方程 3 个未知数 → 最小二乘
    """
    distances = np.array(distances)
    
    if np.all(distances == 0):
        print(f'⚠️ {name}: 数据全为 0，请先填入测量值')
        return None
    
    def residuals(cam_pos):
        cx, cy, cz = cam_pos
        predicted = np.sqrt(
            (corners[:, 0] - cx)**2 + 
            (corners[:, 1] - cy)**2 + 
            cz**2
        )
        return predicted - distances
    
    # 初始猜测
    if initial_guess is None:
        initial_guess = [4.1, -5.0, 6.0]  # 近端相机默认
    
    # 约束: z > 0 (相机在地面以上)
    result = least_squares(
        residuals, 
        initial_guess,
        bounds=([-np.inf, -np.inf, 0.1], [np.inf, np.inf, 20.0]),
    )
    
    cx, cy, cz = result.x
    res = result.fun  # residuals
    rmse = np.sqrt(np.mean(res**2))
    
    print(f'\n{"=" * 50}')
    print(f'{name} 相机位置:')
    print(f'  X = {cx:.3f} m  (球场宽度方向, 中线=4.115)')
    print(f'  Y = {cy:.3f} m  (球场长度方向, 底线=0/23.77)')
    print(f'  Z = {cz:.3f} m  (高度)')
    print(f'\n  RMSE = {rmse*100:.1f} cm')
    print(f'  各角残差: {["{:.1f}cm".format(r*100) for r in res]}')
    print(f'{"=" * 50}')
    
    # 验证: 用算出的位置反算斜距，和测量值对比
    print(f'\n  验证 (计算斜距 vs 测量斜距):')
    for i, (corner, d_measured) in enumerate(zip(corners, distances)):
        d_calc = np.sqrt((cx - corner[0])**2 + (cy - corner[1])**2 + cz**2)
        err = abs(d_calc - d_measured)
        print(f'    角{i+1}: 测={d_measured:.3f}m  算={d_calc:.3f}m  差={err*100:.1f}cm {"✅" if err < 0.05 else "⚠️"}')
    
    return result.x

# 求解 cam66
pos66 = solve_camera_position(
    corners, cam66_distances, 'cam66',
    initial_guess=[4.1, -5.0, 6.0]  # 近端相机
)

# 求解 cam68
pos68 = solve_camera_position(
    corners, cam68_distances, 'cam68',
    initial_guess=[4.1, 28.0, 5.5]  # 远端相机
)


cam66 相机位置:
  X = 4.446 m  (球场宽度方向, 中线=4.115)
  Y = 29.049 m  (球场长度方向, 底线=0/23.77)
  Z = 6.830 m  (高度)

  RMSE = 0.1 cm
  各角残差: ['-0.1cm', '0.1cm', '0.0cm', '-0.0cm']

  验证 (计算斜距 vs 测量斜距):
    角1: 测=30.000m  算=29.999m  差=0.1cm ✅
    角2: 测=29.938m  算=29.939m  差=0.1cm ✅
    角3: 测=9.164m  算=9.164m  差=0.0cm ✅
    角4: 测=8.964m  算=8.964m  差=0.0cm ✅

cam68 相机位置:
  X = 4.431 m  (球场宽度方向, 中线=4.115)
  Y = -5.278 m  (球场长度方向, 底线=0/23.77)
  Z = 6.096 m  (高度)

  RMSE = 8.5 cm
  各角残差: ['3.4cm', '-3.3cm', '-11.6cm', '11.6cm']

  验证 (计算斜距 vs 测量斜距):
    角1: 测=8.591m  算=8.625m  差=3.4cm ✅
    角2: 测=8.454m  算=8.421m  差=3.3cm ✅
    角3: 测=29.954m  算=29.838m  差=11.6cm ⚠️
    角4: 测=29.664m  算=29.780m  差=11.6cm ⚠️


In [ ]:
# ============================================================
# 3D 可视化 (带墙面参照物)
# ============================================================
import plotly.graph_objects as go

fig = go.Figure()

# ---- 场地参数 ----
SX_MIN, SX_MAX = 1.37, 6.86
DOUBLES_W = 8.23  # 双打场宽
CL = 23.77
CX = 4.115  # 球场中心 X
NET_Y = 11.885

# ---- 场馆墙面参数 (★ 根据实际场地调整) ----
WALL_NEAR_Y = -6.0    # 近端墙 Y 坐标 (cam68 靠这面墙)
WALL_FAR_Y  = 30.0    # 远端墙 Y 坐标
WALL_LEFT_X = -1.0    # 左侧墙 X 坐标
WALL_RIGHT_X = 9.23   # 右侧墙 X 坐标
WALL_HEIGHT  = 9.0    # 墙高 (米)

# ---- 墙面 (半透明) ----
wall_color = 'rgba(150,150,170,0.15)'
wall_line = dict(color='rgba(150,150,170,0.4)', width=1)

# 近端墙 (cam68 靠这面)
fig.add_trace(go.Mesh3d(
    x=[WALL_LEFT_X, WALL_RIGHT_X, WALL_RIGHT_X, WALL_LEFT_X],
    y=[WALL_NEAR_Y]*4,
    z=[0, 0, WALL_HEIGHT, WALL_HEIGHT],
    i=[0,0], j=[1,2], k=[2,3],
    color='rgba(100,150,255,0.12)', name='近端墙 (cam68)',
    showlegend=True, hoverinfo='name',
))
# 近端墙边框
for x0, x1, z0, z1 in [(WALL_LEFT_X,WALL_RIGHT_X,0,0),
                         (WALL_LEFT_X,WALL_RIGHT_X,WALL_HEIGHT,WALL_HEIGHT),
                         (WALL_LEFT_X,WALL_LEFT_X,0,WALL_HEIGHT),
                         (WALL_RIGHT_X,WALL_RIGHT_X,0,WALL_HEIGHT)]:
    fig.add_trace(go.Scatter3d(
        x=[x0,x1], y=[WALL_NEAR_Y,WALL_NEAR_Y], z=[z0,z1],
        mode='lines', line=dict(color='rgba(100,150,255,0.5)', width=2),
        showlegend=False, hoverinfo='skip',
    ))

# 远端墙
fig.add_trace(go.Mesh3d(
    x=[WALL_LEFT_X, WALL_RIGHT_X, WALL_RIGHT_X, WALL_LEFT_X],
    y=[WALL_FAR_Y]*4,
    z=[0, 0, WALL_HEIGHT, WALL_HEIGHT],
    i=[0,0], j=[1,2], k=[2,3],
    color='rgba(170,170,170,0.10)', name='远端墙',
    showlegend=True, hoverinfo='name',
))
for x0, x1, z0, z1 in [(WALL_LEFT_X,WALL_RIGHT_X,0,0),
                         (WALL_LEFT_X,WALL_RIGHT_X,WALL_HEIGHT,WALL_HEIGHT),
                         (WALL_LEFT_X,WALL_LEFT_X,0,WALL_HEIGHT),
                         (WALL_RIGHT_X,WALL_RIGHT_X,0,WALL_HEIGHT)]:
    fig.add_trace(go.Scatter3d(
        x=[x0,x1], y=[WALL_FAR_Y,WALL_FAR_Y], z=[z0,z1],
        mode='lines', line=dict(color='rgba(170,170,170,0.4)', width=2),
        showlegend=False, hoverinfo='skip',
    ))

# 左侧墙
fig.add_trace(go.Mesh3d(
    x=[WALL_LEFT_X]*4,
    y=[WALL_NEAR_Y, WALL_FAR_Y, WALL_FAR_Y, WALL_NEAR_Y],
    z=[0, 0, WALL_HEIGHT, WALL_HEIGHT],
    i=[0,0], j=[1,2], k=[2,3],
    color='rgba(170,170,170,0.08)', name='左侧墙',
    showlegend=True, hoverinfo='name',
))

# 右侧墙
fig.add_trace(go.Mesh3d(
    x=[WALL_RIGHT_X]*4,
    y=[WALL_NEAR_Y, WALL_FAR_Y, WALL_FAR_Y, WALL_NEAR_Y],
    z=[0, 0, WALL_HEIGHT, WALL_HEIGHT],
    i=[0,0], j=[1,2], k=[2,3],
    color='rgba(170,170,170,0.08)', name='右侧墙',
    showlegend=True, hoverinfo='name',
))

# ---- 地面 ----
fig.add_trace(go.Mesh3d(
    x=[WALL_LEFT_X, WALL_RIGHT_X, WALL_RIGHT_X, WALL_LEFT_X],
    y=[WALL_NEAR_Y, WALL_NEAR_Y, WALL_FAR_Y, WALL_FAR_Y],
    z=[0]*4,
    i=[0,0], j=[1,2], k=[2,3],
    color='rgba(50,80,50,0.15)', name='地面',
    showlegend=False, hoverinfo='skip',
))

# ---- 球场线 ----
court_x = [SX_MIN, SX_MAX, SX_MAX, SX_MIN, SX_MIN]
court_y = [0, 0, CL, CL, 0]
fig.add_trace(go.Scatter3d(
    x=court_x, y=court_y, z=[0]*5,
    mode='lines', line=dict(color='white', width=3), name='单打场'
))

# 双打边线
fig.add_trace(go.Scatter3d(
    x=[0, 0, DOUBLES_W, DOUBLES_W, 0], y=[0, CL, CL, 0, 0], z=[0]*5,
    mode='lines', line=dict(color='rgba(255,255,255,0.3)', width=2), name='双打场'
))

# 中线 & 发球线
fig.add_trace(go.Scatter3d(
    x=[CX, CX], y=[5.485, 18.285], z=[0, 0],
    mode='lines', line=dict(color='rgba(255,255,255,0.5)', width=2),
    showlegend=False,
))
for sy in [5.485, 18.285]:
    fig.add_trace(go.Scatter3d(
        x=[SX_MIN, SX_MAX], y=[sy, sy], z=[0, 0],
        mode='lines', line=dict(color='rgba(255,255,255,0.5)', width=2),
        showlegend=False,
    ))

# ---- 网 ----
fig.add_trace(go.Scatter3d(
    x=[SX_MIN-0.5, SX_MAX+0.5], y=[NET_Y, NET_Y], z=[0.914, 0.914],
    mode='lines', line=dict(color='gray', width=3), name='Net'
))

# ---- 4 个角点 ----
fig.add_trace(go.Scatter3d(
    x=corners[:, 0], y=corners[:, 1], z=corners[:, 2],
    mode='markers+text',
    marker=dict(size=6, color='yellow'),
    text=['角1(近左)', '角2(近右)', '角3(远左)', '角4(远右)'],
    textposition='top center',
    name='测量角点'
))

# ---- 相机位置 ----
if pos66 is not None:
    fig.add_trace(go.Scatter3d(
        x=[pos66[0]], y=[pos66[1]], z=[pos66[2]],
        mode='markers+text',
        marker=dict(size=10, color='red', symbol='diamond'),
        text=[f'cam66<br>({pos66[0]:.2f}, {pos66[1]:.2f}, {pos66[2]:.2f})'],
        textposition='top center',
        name='cam66'
    ))
    for i, c in enumerate(corners):
        fig.add_trace(go.Scatter3d(
            x=[pos66[0], c[0]], y=[pos66[1], c[1]], z=[pos66[2], 0],
            mode='lines', line=dict(color='red', width=1, dash='dot'),
            showlegend=False
        ))
    # cam66 投影到墙面的辅助线
    fig.add_trace(go.Scatter3d(
        x=[pos66[0], pos66[0]], y=[pos66[1], WALL_FAR_Y], z=[pos66[2], pos66[2]],
        mode='lines', line=dict(color='rgba(255,100,100,0.4)', width=1, dash='dash'),
        showlegend=False,
    ))

if pos68 is not None:
    fig.add_trace(go.Scatter3d(
        x=[pos68[0]], y=[pos68[1]], z=[pos68[2]],
        mode='markers+text',
        marker=dict(size=10, color='blue', symbol='diamond'),
        text=[f'cam68<br>({pos68[0]:.2f}, {pos68[1]:.2f}, {pos68[2]:.2f})'],
        textposition='top center',
        name='cam68'
    ))
    for i, c in enumerate(corners):
        fig.add_trace(go.Scatter3d(
            x=[pos68[0], c[0]], y=[pos68[1], c[1]], z=[pos68[2], 0],
            mode='lines', line=dict(color='blue', width=1, dash='dot'),
            showlegend=False
        ))
    # cam68 到近端墙的距离标注
    dist_to_wall = abs(pos68[1] - WALL_NEAR_Y)
    fig.add_trace(go.Scatter3d(
        x=[pos68[0], pos68[0]], y=[pos68[1], WALL_NEAR_Y], z=[pos68[2], pos68[2]],
        mode='lines+text', line=dict(color='rgba(100,150,255,0.6)', width=2, dash='dash'),
        text=['', f'距墙 {dist_to_wall:.2f}m'],
        textposition='middle right',
        showlegend=False,
    ))

# ---- 标注: cam68 在墙上的投影点 ----
if pos68 is not None:
    fig.add_trace(go.Scatter3d(
        x=[pos68[0]], y=[WALL_NEAR_Y], z=[pos68[2]],
        mode='markers+text',
        marker=dict(size=5, color='cyan', symbol='x'),
        text=[f'墙上投影<br>Z={pos68[2]:.2f}m'],
        textposition='bottom center',
        showlegend=False,
    ))

fig.update_layout(
    title='相机位置 3D 可视化 (含墙面参照)',
    scene=dict(
        xaxis_title='X (m)', yaxis_title='Y (m)', zaxis_title='Z (m)',
        aspectmode='data',
        camera=dict(eye=dict(x=1.5, y=-1.5, z=1.0)),
        xaxis=dict(backgroundcolor='rgba(0,0,0,0)'),
        yaxis=dict(backgroundcolor='rgba(0,0,0,0)'),
        zaxis=dict(backgroundcolor='rgba(0,0,0,0)'),
    ),
    width=1000, height=750,
    paper_bgcolor='#1a1a2e',
    font=dict(color='white'),
)
fig.show()

# ---- 打印 cam68 与墙的关系 ----
if pos68 is not None:
    print(f'\n📏 cam68 与近端墙 (Y={WALL_NEAR_Y}) 的距离: {abs(pos68[1] - WALL_NEAR_Y):.2f} m')
    print(f'   如果 cam68 是靠墙安装的，墙的 Y 坐标应该 ≈ {pos68[1]:.2f}')
    print(f'   → 请将 WALL_NEAR_Y 调整为 {pos68[1]:.1f} 或实际测量的墙面位置')

In [ ]:
# ============================================================
# 生成 config.yaml 更新值
# ============================================================

if pos66 is not None and pos68 is not None:
    baseline = np.sqrt(np.sum((pos66 - pos68)**2))
    
    print('将以下值更新到 config.yaml:')
    print()
    print('cameras:')
    print('  cam66:')
    print(f'    position_3d: [{pos66[0]:.4f}, {pos66[1]:.4f}, {pos66[2]:.4f}]')
    print('  cam68:')
    print(f'    position_3d: [{pos68[0]:.4f}, {pos68[1]:.4f}, {pos68[2]:.4f}]')
    print()
    print(f'基线距离: {baseline:.2f} m')
    print(f'cam66 高度: {pos66[2]:.2f} m')
    print(f'cam68 高度: {pos68[2]:.2f} m')
    print(f'cam66 X偏移(离中线): {pos66[0] - 4.115:.3f} m')
    print(f'cam68 X偏移(离中线): {pos68[0] - 4.115:.3f} m')
    
    # 和之前的估计值对比
    print()
    print('对比之前的估计值:')
    print(f'  cam66: 之前=[4.094, -5.21, 6.2] 现在=[{pos66[0]:.3f}, {pos66[1]:.3f}, {pos66[2]:.3f}]')
    print(f'  cam68: 之前=[4.115, 28.97, 5.2] 现在=[{pos68[0]:.3f}, {pos68[1]:.3f}, {pos68[2]:.3f}]')
else:
    print('请先填入测量数据并运行求解 cell')

In [7]:
# ============================================================
# 额外: 如果你还量了其他点（发球线角、网柱等），可以加在这里
# ============================================================

# 更多测量点（可选，提高精度）
extra_points = {
    # 'net_left':  {'world': [1.37, 11.885, 0], 'cam66_dist': 0.0, 'cam68_dist': 0.0},
    # 'net_right': {'world': [6.86, 11.885, 0], 'cam66_dist': 0.0, 'cam68_dist': 0.0},
    # 'serve_near_left':  {'world': [1.37, 5.485, 0], 'cam66_dist': 0.0, 'cam68_dist': 0.0},
    # 'serve_near_right': {'world': [6.86, 5.485, 0], 'cam66_dist': 0.0, 'cam68_dist': 0.0},
}

if extra_points:
    all_corners = np.vstack([corners] + [np.array(v['world']).reshape(1,3) for v in extra_points.values()])
    all_d66 = cam66_distances + [v['cam66_dist'] for v in extra_points.values()]
    all_d68 = cam68_distances + [v['cam68_dist'] for v in extra_points.values()]
    
    print('加入额外测量点后重新求解:')
    pos66_v2 = solve_camera_position(all_corners, all_d66, 'cam66 (enhanced)', initial_guess=list(pos66))
    pos68_v2 = solve_camera_position(all_corners, all_d68, 'cam68 (enhanced)', initial_guess=list(pos68))
else:
    print('取消注释 extra_points 中的行并填入数据，可以提高定位精度')

取消注释 extra_points 中的行并填入数据，可以提高定位精度
